# 04 — Retrieval: dense vs BM25 vs hybrid vs reranked

Stage 4 (`app/retrieval.py`) on the **real** corpus with the **real** models. The corpus was
ingested by `uv run python -m evals.run_eval --config all` into the shared `structure`
collection (whole PDFs, structure-aware chunks). For 10 golden-set questions from the
`tune` split (never `heldout`) we compare four retrieval setups side by side, work RRF out
by hand for one question, time the reranker, and look at the scores the refusal threshold
will use. ✓ marks a chunk that matches the question's labelled document + page.

In [1]:
# Run from the project root so app/ imports, .env and corpus/ paths work.
import os, sys, time, warnings
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ["TQDM_DISABLE"] = "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")

import pandas as pd
pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 200)
print("project root:", Path.cwd().name)

project root: fbr-di-assistant


## 1. Load the models and connect to Qdrant

The same objects the API loads once in its lifespan: BGE-M3, BM25 and the
bge-reranker-v2-m3 cross-encoder.

In [2]:
from qdrant_client import AsyncQdrantClient
from app.config import settings
from app.ingestion.embed import create_embedders
from app.retrieval import Retrievers, RetrievalConfig, create_reranker, retrieve
from evals.config import load_config
from evals.ingest_corpus import EVAL_OWNER_ID

started = time.perf_counter()
dense, sparse = create_embedders()
reranker = create_reranker()
print(f"models loaded in {time.perf_counter() - started:.0f} s")

qdrant = AsyncQdrantClient(url=settings.qdrant_url, timeout=60)
retrievers = Retrievers(dense=dense, sparse=sparse, reranker=reranker, qdrant=qdrant)
COLLECTION = load_config("final").collection_name()
points = (await qdrant.count(COLLECTION, exact=True)).count
print(f"collection {COLLECTION}: {points} chunks")

models loaded in 15 s
collection eval_structure_3fa86c58: 2688 chunks


## 2. Ten tune questions

One per category (the first in the `tune` split), plus a second exact-code question,
because exact codes are where dense and BM25 differ most.

In [3]:
from evals.golden_set import load_golden_set

tune = [q for q in load_golden_set() if q.split == "tune"]
picked, seen = [], set()
for q in tune:
    if q.category not in seen:
        picked.append(q); seen.add(q.category)
picked.append(next(q for q in tune if q.category == "exact_code" and q not in picked))
pd.DataFrame(
    [{"id": q.id, "category": q.category.value, "question": q.question,
      "expected": "; ".join(f"{s.doc} p.{s.pages}" for s in q.sources) or "(unanswerable)"}
     for q in picked]
)

,id,category,question,expected
0,q001,exact_code,Our ERP got error 0052 back from postinvoicedata. What's wrong wit...,di_api_technical_spec_v1.12.pdf p.[38]
1,q008,field_rule,Is buyerNTNCNIC mandatory when the buyer is unregistered?,di_api_technical_spec_v1.12.pdf p.[10]
2,q015,legal_rule,"How long do we have to keep the e-invoices we issue, and in what f...",sales_tax_rules_2006_upto_2026-07-31.pdf p.[108]
3,q022,table_matrix,We're a textile manufacturer. Which sandbox scenarios apply to us?,di_api_technical_spec_v1.12.pdf p.[47]
4,q027,multi_part,The law and the PRAL spec both describe the QR code on the invoice...,sales_tax_rules_2006_upto_2026-07-31.pdf p.[107]; di_api_technical...
5,q032,screenshot,How much can a licensed integrator other than PRAL charge us for d...,di_user_manual_v1.5.pdf p.[7]
6,q035,version_sensitive,How many server IP addresses can we enter in the IP whitelisting f...,di_user_manual_v1.5.pdf p.[9]
7,q038,reworded_faq,Should our system create the digital invoice when we bill the cust...,faqs.jsonl p.[20]; sales_tax_act_1990_upto_2026-06-30.pdf p.[24]
8,q043,unanswerable,What is the income tax rate for a salaried individual earning Rs. ...,(unanswerable)
9,q002,exact_code,What does validation error 0401 mean and what usually causes it?,di_api_technical_spec_v1.12.pdf p.[41]


## 3. The four setups

| Setup | What runs |
| --- | --- |
| dense | BGE-M3 only (meaning) |
| bm25 | BM25 only (exact words, IDF from Qdrant) |
| hybrid | both, fused by RRF (k = 60) |
| reranked | hybrid → cross-encoder → exact-code + authority boosts (the `final` config) |

In [4]:
from evals.metrics import RetrievedRef, is_hit

SETUPS = {
    "dense": RetrievalConfig(mode="dense", rerank=False, exact_boost=False, authority_boost=False),
    "bm25": RetrievalConfig(mode="bm25", rerank=False, exact_boost=False, authority_boost=False),
    "hybrid": RetrievalConfig(mode="hybrid", rerank=False, exact_boost=False, authority_boost=False),
    "reranked": RetrievalConfig(),
}

def hit(chunk, question):
    ref = RetrievedRef(chunk.filename, chunk.page_start, chunk.page_end)
    return any(is_hit(ref, label) for label in question.labels())

async def run(question, setup):
    return await retrieve(question.question, EVAL_OWNER_ID, retrievers, SETUPS[setup], COLLECTION)

results = {}
for q in picked:
    for setup in SETUPS:
        results[q.id, setup] = await run(q, setup)
print("done:", len(results), "searches")

done: 40 searches


### First correct rank per question (lower is better; — = not in the top 6)

In [5]:
def first_hit(question, result):
    return next((c.rank for c in result.chunks if hit(c, question)), None)

table = pd.DataFrame(
    [{"id": q.id, "category": q.category.value,
      **{s: first_hit(q, results[q.id, s]) for s in SETUPS}} for q in picked]
).set_index("id")
table.fillna("—")

,category,dense,bm25,hybrid,reranked
id,,,,,
q001,exact_code,2.0,2.0,3.0,2.0
q008,field_rule,—,1.0,—,1.0
q015,legal_rule,1.0,—,4.0,1.0
q022,table_matrix,1.0,1.0,1.0,2.0
q027,multi_part,1.0,4.0,1.0,1.0
q032,screenshot,—,2.0,3.0,1.0
q035,version_sensitive,6.0,3.0,4.0,3.0
q038,reworded_faq,2.0,—,—,1.0
q043,unanswerable,—,—,—,—


### Top 5 per setup for each question

`sim` is the cosine similarity (what the chunks panel shows), `rerank` the raw
cross-encoder score (0-1), `⚑` an exact code match.

In [6]:
from IPython.display import Markdown, display

def top5(question, setup):
    rows = []
    for c in results[question.id, setup].chunks[:5]:
        rows.append({
            "setup": setup, "#": c.rank, "✓": "✓" if hit(c, question) else "",
            "doc": c.filename[:28], "page": c.page_start,
            "section": " > ".join(c.section_path)[-45:],
            "sim": round(c.similarity, 3),
            "rerank": None if c.rerank_score is None else round(c.rerank_score, 3),
            "⚑": "⚑" if c.exact_match else "",
        })
    return rows

for q in picked:
    display(Markdown(f"**{q.id}** ({q.category.value}) — {q.question}"))
    display(pd.DataFrame([row for s in SETUPS for row in top5(q, s)]).set_index(["setup", "#"]))

**q001** (exact_code) — Our ERP got error 0052 back from postinvoicedata. What's wrong with the invoice?

✓                           doc  page                                        section    sim  rerank  ⚑
setup    #                                                                                                        
dense    1     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.657     NaN   
         2  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.642     NaN   
         3  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.640     NaN   
         4  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.635     NaN   
         5  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.628     NaN   
bm25     1     di_api_technical_spec_v1.12.    13  Post > 4.1.4. Sample JSON Invalid Response 1:  0.612     NaN   
         2  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.601     NaN   
         3          di_user_manual_v1.5.pdf    18  nefits of using the Digital Invoicing System?  0.519     NaN   
         4     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.657     NaN   
         5     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.587     NaN   
hybrid   1     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.657     NaN   
         2     di_api_technical_spec_v1.12.    13  Post > 4.1.4. Sample JSON Invalid Response 1:  0.612     NaN   
         3  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.601     NaN   
         4     di_api_technical_spec_v1.12.    36  l Invoicing System image to print on receipts  0.606     NaN   
         5          di_user_manual_v1.5.pdf    20              20. What if a test invoice fails?  0.614     NaN   
reranked 1     di_api_technical_spec_v1.12.    13  Post > 4.1.4. Sample JSON Invalid Response 1:  0.612   0.006  ⚑
         2  ✓  di_api_technical_spec_v1.12.    38                           7. Sales Error Codes  0.601   0.005  ⚑
         3     di_api_technical_spec_v1.12.     9  t > 4.1.2. Sample JSON Data (For Production):  0.561   0.108   
         4     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.657   0.020   
         5          di_user_manual_v1.5.pdf    20              20. What if a test invoice fails?  0.614   0.018

**q008** (field_rule) — Is buyerNTNCNIC mandatory when the buyer is unregistered?

✓                           doc  page                                        section    sim  rerank  ⚑
setup    #                                                                                                        
dense    1                       faqs.jsonl    18                                         FAQ 18  0.655     NaN   
         2     sales_tax_rules_2006_upto_20   175                                          STR-7  0.586     NaN   
         3     sales_tax_rules_2006_upto_20   171                                          STR-1  0.577     NaN   
         4     sales_tax_rules_2006_upto_20   174                                          STR-1  0.570     NaN   
         5                       faqs.jsonl     3                                          FAQ 3  0.567     NaN   
bm25     1  ✓  di_api_technical_spec_v1.12.    10  t > 4.1.2. Sample JSON Data (For Production):  0.498     NaN   
         2     di_api_technical_spec_v1.12.    18  e > 4.2.2. Sample JSON Data (For Production):  0.520     NaN   
         3     di_api_technical_spec_v1.12.    17  e > 4.2.2. Sample JSON Data (For Production):  0.460     NaN   
         4  ✓  di_api_technical_spec_v1.12.     9  t > 4.1.2. Sample JSON Data (For Production):  0.448     NaN   
         5                       faqs.jsonl    18                                         FAQ 18  0.655     NaN   
hybrid   1                       faqs.jsonl    18                                         FAQ 18  0.655     NaN   
         2     sales_tax_rules_2006_upto_20   175                                          STR-7  0.586     NaN   
         3                       faqs.jsonl     3                                          FAQ 3  0.567     NaN   
         4     di_api_technical_spec_v1.12.    18  e > 4.2.2. Sample JSON Data (For Production):  0.520     NaN   
         5          di_user_manual_v1.5.pdf    16                                    2. New Case  0.544     NaN   
reranked 1  ✓  di_api_technical_spec_v1.12.    10  t > 4.1.2. Sample JSON Data (For Production):  0.498   0.905  ⚑
         2     di_api_technical_spec_v1.12.    18  e > 4.2.2. Sample JSON Data (For Production):  0.520   0.900  ⚑
         3  ✓  di_api_technical_spec_v1.12.     9  t > 4.1.2. Sample JSON Data (For Production):  0.448   0.634  ⚑
         4     di_api_technical_spec_v1.12.    17  e > 4.2.2. Sample JSON Data (For Production):  0.460   0.620  ⚑
         5     sales_tax_rules_2006_upto_20   175                                          STR-7  0.586   0.563

**q015** (legal_rule) — How long do we have to keep the e-invoices we issue, and in what form?

✓                           doc  page                                        section    sim  rerank ⚑
setup    #                                                                                                       
dense    1  ✓  sales_tax_rules_2006_upto_20   108  INTEGRATION OF REGISTERED PERSONS > Rule 150S  0.718     NaN  
         2     sro_69(I)_2025_digital_invoi     4                                      Rule 150Q  0.697     NaN  
         3     sro_69(I)_2025_digital_invoi     8                                     Rule 150XJ  0.648     NaN  
         4     sales_tax_rules_2006_upto_20   111  NTEGRATION OF REGISTERED PERSONS > Rule 150XJ  0.637     NaN  
         5                       faqs.jsonl    22                                         FAQ 22  0.633     NaN  
bm25     1     sales_tax_rules_2006_upto_20    66  ERSONS AND PRIVILEGED ORGANISATIONS > Rule 52  0.451     NaN  
         2     sales_tax_rules_2006_upto_20    83                         CHAPTER X-A > Rule 69E  0.503     NaN  
         3     sales_tax_act_1990_upto_2026    58               Sales Tax Act, 1990 > Section 22  0.516     NaN  
         4     sales_tax_rules_2006_upto_20   106  INTEGRATION OF REGISTERED PERSONS > Rule 150R  0.582     NaN  
         5     sro_69(I)_2025_digital_invoi     2                                      Rule 150Q  0.630     NaN  
hybrid   1     sro_69(I)_2025_digital_invoi     2                                      Rule 150Q  0.630     NaN  
         2                       faqs.jsonl    22                                         FAQ 22  0.633     NaN  
         3                       faqs.jsonl    17                                         FAQ 17  0.585     NaN  
         4  ✓  sales_tax_rules_2006_upto_20   108  INTEGRATION OF REGISTERED PERSONS > Rule 150S  0.718     NaN  
         5     sales_tax_rules_2006_upto_20    66  ERSONS AND PRIVILEGED ORGANISATIONS > Rule 52  0.451     NaN  
reranked 1  ✓  sales_tax_rules_2006_upto_20   108  INTEGRATION OF REGISTERED PERSONS > Rule 150S  0.718   0.821  
         2     sro_69(I)_2025_digital_invoi     4                                      Rule 150Q  0.697   0.578  
         3     sales_tax_rules_2006_upto_20   111  NTEGRATION OF REGISTERED PERSONS > Rule 150XJ  0.637   0.143  
         4     sro_69(I)_2025_digital_invoi     2                                      Rule 150Q  0.630   0.079  
         5     sro_69(I)_2025_digital_invoi     8                                     Rule 150XJ  0.648   0.058

... (10 outputs left out)

**q043** (unanswerable) — What is the income tax rate for a salaried individual earning Rs. 2 million a year in tax year 2026?

✓                           doc  page                                        section    sim  rerank ⚑
setup    #                                                                                                      
dense    1    sales_tax_act_1990_upto_2026   218                                            217  0.544     NaN  
         2    sales_tax_rules_2006_upto_20   175                                          STR-7  0.511     NaN  
         3    sales_tax_rules_2006_upto_20    74                                             2.  0.509     NaN  
         4    sales_tax_act_1990_upto_2026   218                                            217  0.498     NaN  
         5    sales_tax_act_1990_upto_2026   212                        Sales Tax Act, 1990 211  0.496     NaN  
bm25     1    sales_tax_rules_2006_upto_20   164                                          STR-1  0.437     NaN  
         2       iris_dashboard_manual.pdf     2                                                 0.433     NaN  
         3    sales_tax_act_1990_upto_2026   220                                            219  0.486     NaN  
         4    sales_tax_rules_2006_upto_20   110  NTEGRATION OF REGISTERED PERSONS > Rule 150XH  0.430     NaN  
         5    sro_69(I)_2025_digital_invoi     7                                     Rule 150XH  0.407     NaN  
hybrid   1    sales_tax_act_1990_upto_2026   220                                            219  0.486     NaN  
         2    sales_tax_rules_2006_upto_20   175                                          STR-7  0.511     NaN  
         3    sales_tax_act_1990_upto_2026    22                            Sales Tax Act, 1990  0.477     NaN  
         4    sales_tax_rules_2006_upto_20    27  ORYREGISTRATION AND DE- REGISTRATION > Rule 5  0.481     NaN  
         5    sales_tax_act_1990_upto_2026   113              Sales Tax Act, 1990 > Section 47A  0.472     NaN  
reranked 1    sales_tax_act_1990_upto_2026   212                        Sales Tax Act, 1990 211  0.496   0.132  
         2    sales_tax_act_1990_upto_2026   220                                            219  0.486   0.013  
         3    sales_tax_rules_2006_upto_20   164                                          STR-1  0.437   0.013  
         4    sales_tax_act_1990_upto_2026    22                            Sales Tax Act, 1990  0.477   0.011  
         5    sales_tax_act_1990_upto_2026   218                                            217  0.498   0.004

**q002** (exact_code) — What does validation error 0401 mean and what usually causes it?

✓                           doc  page                                        section    sim  rerank  ⚑
setup    #                                                                                                        
dense    1  ✓  di_api_technical_spec_v1.12.    41                           7. Sales Error Codes  0.573     NaN   
         2     di_api_technical_spec_v1.12.    13  Post > 4.1.4. Sample JSON Invalid Response 1:  0.556     NaN   
         3  ✓  di_api_technical_spec_v1.12.    41                           7. Sales Error Codes  0.544     NaN   
         4     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.539     NaN   
         5     di_api_technical_spec_v1.12.    14  Post > 4.1.5. Sample JSON Invalid Response 2:  0.539     NaN   
bm25     1  ✓  di_api_technical_spec_v1.12.    41                           7. Sales Error Codes  0.573     NaN   
         2     sales_tax_act_1990_upto_2026   106              Sales Tax Act, 1990 > Section 40D  0.349     NaN   
         3     sales_tax_act_1990_upto_2026     7                Sales Tax Act, 1990 > Section 2  0.316     NaN   
         4     sales_tax_act_1990_upto_2026    20                            Sales Tax Act, 1990  0.336     NaN   
         5     sales_tax_act_1990_upto_2026    48              Sales Tax Act, 1990 > Section 11G  0.337     NaN   
hybrid   1  ✓  di_api_technical_spec_v1.12.    41                           7. Sales Error Codes  0.573     NaN   
         2     di_api_technical_spec_v1.12.    20  2.3. Sample Json Response (In Case of Valid):  0.523     NaN   
         3     di_api_technical_spec_v1.12.    22  4. Sample Json Response (In Case of Invalid):  0.531     NaN   
         4     di_api_technical_spec_v1.12.    40                           7. Sales Error Codes  0.527     NaN   
         5     di_api_technical_spec_v1.12.    12  g - Post > 4.1.3. Sample JSON Valid Response:  0.521     NaN   
reranked 1  ✓  di_api_technical_spec_v1.12.    41                           7. Sales Error Codes  0.573   0.178  ⚑
         2     sales_tax_act_1990_upto_2026    20                            Sales Tax Act, 1990  0.336   0.001   
         3     sales_tax_act_1990_upto_2026   106              Sales Tax Act, 1990 > Section 40D  0.349   0.000   
         4     sales_tax_act_1990_upto_2026     7                Sales Tax Act, 1990 > Section 2  0.316   0.000   
         5     sales_tax_act_1990_upto_2026    48              Sales Tax Act, 1990 > Section 11G  0.337   0.000

## 4. RRF by hand

For the first exact-code question we fetch the dense and BM25 lists separately and add
`1 / (60 + rank)` for every list a chunk appears in. Only **ranks** are used: cosine
(0-1) and BM25 (unbounded) scores cannot be compared directly. The top of our manual
fusion should match Qdrant's own hybrid order.

In [7]:
from app.retrieval import embed_query, search_candidates

q = next(q for q in picked if q.category.value == "exact_code")
dense_vec, sparse_vec = await embed_query(q.question, retrievers, "hybrid")
dense_list = await search_candidates(qdrant, COLLECTION, dense_vec, sparse_vec, EVAL_OWNER_ID, SETUPS["dense"])
bm25_list = await search_candidates(qdrant, COLLECTION, dense_vec, sparse_vec, EVAL_OWNER_ID, SETUPS["bm25"])

K = settings.rrf_k
fused = {}
for name, ranked in (("dense", dense_list), ("bm25", bm25_list)):
    for rank, point in enumerate(ranked, start=1):
        entry = fused.setdefault(point.id, {"text": point.payload["text"][:60], "dense": None, "bm25": None})
        entry[name] = rank
for entry in fused.values():
    entry["rrf"] = sum(1 / (K + entry[n]) for n in ("dense", "bm25") if entry[n])

manual = sorted(fused.items(), key=lambda item: item[1]["rrf"], reverse=True)[:8]
qdrant_order = [c.chunk_id for c in results[q.id, "hybrid"].chunks]
print(q.id, q.question)
pd.DataFrame(
    [{"dense rank": e["dense"], "bm25 rank": e["bm25"], "RRF": round(e["rrf"], 5),
      "Qdrant hybrid rank": qdrant_order.index(str(pid)) + 1 if str(pid) in qdrant_order else None,
      "text": e["text"]} for pid, e in manual]
)

q001 Our ERP got error 0052 back from postinvoicedata. What's wrong with the invoice?


,dense rank,bm25 rank,RRF,Qdrant hybrid rank,text
0,1.0,4.0,0.03202,1.0,|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|\n|---|---|---|\n|01
1,15.0,1.0,0.02973,2.0,"{\n""dated"": ""2025-05-13 13:09:05"",\n""validationResponse"": {\n""s"
2,27.0,2.0,0.02762,3.0,|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|\n|---|---|---|\n|00
3,22.0,13.0,0.02589,4.0,|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|\n|---|---|---|\n|00
4,12.0,28.0,0.02525,5.0,Use the error code returned to fix issues. Refer to detailed
5,2.0,NaN,0.01613,6.0,|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|\n|---|---|---|\n|00
6,3.0,NaN,0.01587,NaN,|MESSAGE CODE|MESSAGE_DESC|BRIEF_MSG_DESC|\n|---|---|---|\n|00
7,NaN,3.0,0.01587,NaN,Real-time invoice submission and validation.\nAutomated tax c


## 5. Reranker speed (and `RERANK_BATCH_SIZE`)

The cross-encoder runs once per (question, chunk) pair, so reranking 30 candidates is the
slowest step of retrieval. Timing a few batch sizes on 30 real candidates:

In [8]:
q = picked[0]
dense_vec, sparse_vec = await embed_query(q.question, retrievers, "hybrid")
points30 = await search_candidates(qdrant, COLLECTION, dense_vec, sparse_vec, EVAL_OWNER_ID, SETUPS["hybrid"])
passages = [p.payload["text"] for p in points30]  # all 30 fused candidates
pairs = [[q.question, p] for p in passages]
timings = []
for batch in (1, 4, 8, 16):
    reranker._model.compute_score(pairs[:2], batch_size=batch, max_length=settings.rerank_max_tokens, normalize=True)  # warm-up
    started = time.perf_counter()
    reranker._model.compute_score(pairs, batch_size=batch, max_length=settings.rerank_max_tokens, normalize=True)
    timings.append({"batch_size": batch, "seconds for 30 pairs": round(time.perf_counter() - started, 2)})
pd.DataFrame(timings)

,batch_size,seconds for 30 pairs
0,1,23.41
1,4,32.28
2,8,45.15
3,16,76.37


## 6. Scores the refusal threshold will see

Best **raw** rerank score per question (boosts excluded). Answerable questions should sit
clearly above unanswerable ones; the gap is where `REFUSAL_THRESHOLD` belongs. The full
sweep over the 30 tune questions is in `evals/results/*_final.md`.

In [9]:
pd.DataFrame(
    [{"id": q.id, "answerable": q.answerable, "category": q.category.value,
      "best rerank": round(results[q.id, "reranked"].best_rerank_score or 0, 3),
      f"refuse at {settings.refusal_threshold}": results[q.id, "reranked"].below_threshold}
     for q in picked]
).sort_values("best rerank", ascending=False)

,id,answerable,category,best rerank,refuse at 0.03
3,q022,True,table_matrix,0.976,False
5,q032,True,screenshot,0.958,False
6,q035,True,version_sensitive,0.952,False
1,q008,True,field_rule,0.905,False
2,q015,True,legal_rule,0.821,False
4,q027,True,multi_part,0.630,False
9,q002,True,exact_code,0.178,False
8,q043,False,unanswerable,0.132,False
0,q001,True,exact_code,0.108,False
7,q038,True,reworded_faq,0.058,False


In [10]:
timing_rows = [results[q.id, "reranked"].timings_ms for q in picked]
print("mean ms per step (reranked setup):",
      {step: round(sum(t[step] for t in timing_rows) / len(timing_rows)) for step in timing_rows[0]})
await qdrant.close()

mean ms per step (reranked setup): {'embed': 353, 'search': 66, 'rerank': 17998}
